In [64]:
"""
Data Quality & Variety Testing Script
Validates demo data and creates visualizations
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
import os

In [68]:
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("=" * 80)
print("📊 BRIGHTLINE RCT DATA QUALITY & VARIETY TEST")
print("=" * 80)

📊 BRIGHTLINE RCT DATA QUALITY & VARIETY TEST


In [69]:


# ============================================================================
# LOAD DATA
# ============================================================================

print("\n1️⃣ Loading data...")

try:
    sales_df = pd.read_csv('data/demo/demo_sales.csv')
    transactions_df = pd.read_csv('data/demo/demo_transactions.csv')
    media_df = pd.read_csv('data/demo/demo_media.csv')
    controls_df = pd.read_csv('data/demo/demo_controls.csv')
    
    with open('data/demo/demo_metadata.json', 'r') as f:
        metadata = json.load(f)
    
    print("✅ Data loaded successfully!")
    print(f"   Sales: {len(sales_df):,} records")
    print(f"   Transactions: {len(transactions_df):,} records")
    print(f"   Media: {len(media_df):,} records")
    print(f"   Controls: {len(controls_df):,} records")
    
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    print("   Run 'python generator_demo_data_v4.py' first!")
    exit(1)

# Convert dates
sales_df['date'] = pd.to_datetime(sales_df['date'])
transactions_df['date'] = pd.to_datetime(transactions_df['date'])
media_df['date'] = pd.to_datetime(media_df['date'])
controls_df['date'] = pd.to_datetime(controls_df['date'])

# ============================================================================
# DATA COMPLETENESS CHECK
# ============================================================================

print("\n" + "=" * 80)
print("2️⃣ DATA COMPLETENESS CHECK")
print("=" * 80)

def check_completeness(df, name):
    print(f"\n{name}:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    
    if missing.sum() == 0:
        print("   ✅ No missing values!")
    else:
        print("   ⚠️ Missing values found:")
        for col, pct in missing_pct[missing_pct > 0].items():
            print(f"      {col}: {pct:.2f}%")
    
    return missing.sum() == 0

all_complete = True
all_complete &= check_completeness(sales_df, "Sales Data")
all_complete &= check_completeness(transactions_df, "Transactions Data")
all_complete &= check_completeness(media_df, "Media Data")
all_complete &= check_completeness(controls_df, "Controls Data")

if all_complete:
    print("\n✅ ALL DATA 100% COMPLETE!")

# ============================================================================
# DATA VARIETY ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("3️⃣ DATA VARIETY ANALYSIS")
print("=" * 80)

print("\n📊 Sales Data Variety:")
print(f"   Unique DMAs: {sales_df['geo_id'].nunique()}")
print(f"   Unique weeks: {sales_df['week'].nunique()}")
print(f"   Retail channels: {sales_df['retail_channel'].nunique()}")
print(f"   Promo types: {sales_df['promo_type'].nunique()}")
print(f"   Sales range: ${sales_df['sales_revenue'].min():,.0f} - ${sales_df['sales_revenue'].max():,.0f}")
print(f"   Sales std dev: ${sales_df['sales_revenue'].std():,.0f}")
print(f"   Coefficient of variation: {sales_df['sales_revenue'].std() / sales_df['sales_revenue'].mean():.2%}")

print("\n💳 Transaction Data Variety:")
print(f"   Unique customers: {transactions_df['customer_id'].nunique()}")
print(f"   Age segments: {transactions_df['age_segment'].nunique()}")
print(f"   New customers: {transactions_df['is_new_customer'].sum():,} ({transactions_df['is_new_customer'].mean()*100:.1f}%)")
print(f"   Transaction units range: {transactions_df['units'].min()} - {transactions_df['units'].max()}")
print(f"   Price range: ${transactions_df['price_per_unit'].min():.2f} - ${transactions_df['price_per_unit'].max():.2f}")

print("\n📺 Media Data Variety:")
print(f"   Media channels: {media_df['channel'].nunique()}")
print(f"   Spend range: ${media_df['spend_usd'].min():.0f} - ${media_df['spend_usd'].max():.0f}")
print(f"   Spend std dev: ${media_df['spend_usd'].std():.0f}")

print("\n🎛️ Controls Data Variety:")
print(f"   Holiday weeks: {controls_df['holiday_flag'].sum()} ({controls_df['holiday_flag'].mean()*100:.1f}%)")
print(f"   Promo weeks: {controls_df['promo_flag'].sum()} ({controls_df['promo_flag'].mean()*100:.1f}%)")
print(f"   Competitor promo weeks: {controls_df['competitor_promo'].sum()} ({controls_df['competitor_promo'].mean()*100:.1f}%)")

# ============================================================================
# PERIOD BREAKDOWN
# ============================================================================

print("\n" + "=" * 80)
print("4️⃣ PERIOD BREAKDOWN")
print("=" * 80)

for period, period_name in [('is_pre_period', 'PRE-PERIOD'), 
                            ('is_test_period', 'TEST PERIOD'), 
                            ('is_post_period', 'POST-PERIOD')]:
    
    period_sales = sales_df[sales_df[period] == True]
    period_transactions = transactions_df[transactions_df[period] == True]
    
    treatment_sales = period_sales[period_sales['is_treatment'] == True]['sales_revenue'].mean()
    control_sales = period_sales[period_sales['is_treatment'] == False]['sales_revenue'].mean()
    
    lift = ((treatment_sales - control_sales) / control_sales) * 100 if control_sales > 0 else 0
    
    print(f"\n{period_name} (Weeks {period_sales['week'].min()}-{period_sales['week'].max()}):")
    print(f"   Records: {len(period_sales):,}")
    print(f"   Treatment avg: ${treatment_sales:,.0f}")
    print(f"   Control avg: ${control_sales:,.0f}")
    print(f"   Lift: {lift:+.1f}%")
    
    if len(period_transactions) > 0:
        print(f"   Transactions: {len(period_transactions):,}")
        print(f"   New customers: {period_transactions['is_new_customer'].sum():,}")

# ============================================================================
# CUSTOMER BEHAVIOR ANALYSIS
# ============================================================================

print("\n" + "=" * 80)
print("5️⃣ CUSTOMER BEHAVIOR ANALYSIS")
print("=" * 80)

customer_purchases = transactions_df.groupby('customer_id').size()
print(f"\n📈 Purchase Frequency Distribution:")
print(f"   1 purchase: {(customer_purchases == 1).sum():,} customers ({(customer_purchases == 1).mean()*100:.1f}%)")
print(f"   2 purchases: {(customer_purchases == 2).sum():,} customers ({(customer_purchases == 2).mean()*100:.1f}%)")
print(f"   3+ purchases: {(customer_purchases >= 3).sum():,} customers ({(customer_purchases >= 3).mean()*100:.1f}%)")
print(f"   Max purchases: {customer_purchases.max()} by one customer")
print(f"   Avg purchases per customer: {customer_purchases.mean():.2f}")

# Age segment analysis
print(f"\n👥 Age Segment Distribution:")
age_dist = transactions_df['age_segment'].value_counts().sort_index()
for age, count in age_dist.items():
    pct = count / len(transactions_df) * 100
    print(f"   {age}: {count:,} transactions ({pct:.1f}%)")

# ============================================================================
# CREATE VISUALIZATIONS
# ============================================================================

print("\n" + "=" * 80)
print("6️⃣ GENERATING VISUALIZATIONS")
print("=" * 80)

os.makedirs('outputs/data_quality', exist_ok=True)

# ============================================================================
# VISUALIZATION 1: Sales Over Time by Period
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Plot treatment
treatment_sales = sales_df[sales_df['is_treatment'] == True].groupby('week')['sales_revenue'].mean()
control_sales = sales_df[sales_df['is_treatment'] == False].groupby('week')['sales_revenue'].mean()

ax.plot(treatment_sales.index, treatment_sales.values, 
       label='Treatment', linewidth=2.5, color='#006E74', marker='o', markersize=4)
ax.plot(control_sales.index, control_sales.values, 
       label='Control', linewidth=2.5, color='#FF6B00', marker='s', markersize=4)

# Add period shading
pre_end = metadata['pre_period_weeks']
test_end = pre_end + metadata['test_period_weeks']

ax.axvspan(0, pre_end, alpha=0.1, color='gray', label='Pre-Period')
ax.axvspan(pre_end, test_end, alpha=0.15, color='green', label='Test Period')
ax.axvspan(test_end, 52, alpha=0.15, color='blue', label='Post-Period')

# Add vertical lines
ax.axvline(pre_end, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
ax.axvline(test_end, color='black', linestyle='--', linewidth=1.5, alpha=0.5)

ax.set_xlabel('Week', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Sales Revenue ($)', fontsize=12, fontweight='bold')
ax.set_title('Sales Over Time: Treatment vs Control by Period', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('outputs/data_quality/1_sales_over_time.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 1_sales_over_time.png")
plt.close()

# ============================================================================
# VISUALIZATION 2: Promo Type Distribution by Period
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

periods = [
    ('is_pre_period', 'Pre-Period', axes[0]),
    ('is_test_period', 'Test Period', axes[1]),
    ('is_post_period', 'Post-Period', axes[2])
]

for period_flag, period_name, ax in periods:
    period_data = sales_df[sales_df[period_flag] == True]
    promo_counts = period_data['promo_type'].value_counts()
    
    colors = {'None': '#E8E8E8', 'Visibility': '#0097AC', 'Price': '#FF6B00'}
    bar_colors = [colors.get(x, '#666666') for x in promo_counts.index]
    
    ax.bar(promo_counts.index, promo_counts.values, color=bar_colors, edgecolor='black', linewidth=1.5)
    ax.set_title(period_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count', fontsize=10)
    ax.set_xlabel('Promo Type', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    total = promo_counts.sum()
    for i, (idx, val) in enumerate(promo_counts.items()):
        pct = val / total * 100
        ax.text(i, val + 20, f'{pct:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.suptitle('Promo Type Distribution Across Periods', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/data_quality/2_promo_distribution.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 2_promo_distribution.png")
plt.close()

# ============================================================================
# VISUALIZATION 3: Sales Distribution by Channel & Period
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

channels = ['Store', 'Omnichannel', 'Online_Specialty']
periods_for_plot = ['Pre', 'Test', 'Post']
period_flags = ['is_pre_period', 'is_test_period', 'is_post_period']

x = np.arange(len(channels))
width = 0.25

for i, (period_name, period_flag) in enumerate(zip(periods_for_plot, period_flags)):
    period_data = sales_df[sales_df[period_flag] == True]
    means = [period_data[period_data['retail_channel'] == ch]['sales_revenue'].mean() for ch in channels]
    
    colors = ['#006E74', '#0097AC', '#FF6B00']
    ax.bar(x + i*width, means, width, label=period_name, color=colors[i], edgecolor='black', linewidth=1)

ax.set_xlabel('Retail Channel', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Sales ($)', fontsize=12, fontweight='bold')
ax.set_title('Average Sales by Channel Across Periods', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(channels)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('outputs/data_quality/3_sales_by_channel.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 3_sales_by_channel.png")
plt.close()

# ============================================================================
# VISUALIZATION 4: Customer Purchase Frequency
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Purchase frequency histogram
purchase_freq = transactions_df.groupby('customer_id').size()
ax1.hist(purchase_freq, bins=range(1, purchase_freq.max()+2), 
        color='#0097AC', edgecolor='black', linewidth=1.5, alpha=0.8)
ax1.set_xlabel('Number of Purchases', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Customers', fontsize=11, fontweight='bold')
ax1.set_title('Customer Purchase Frequency Distribution', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Age segment distribution
age_dist = transactions_df['age_segment'].value_counts().sort_index()
colors = plt.cm.viridis(np.linspace(0, 1, len(age_dist)))
ax2.bar(range(len(age_dist)), age_dist.values, color=colors, edgecolor='black', linewidth=1.5)
ax2.set_xticks(range(len(age_dist)))
ax2.set_xticklabels(age_dist.index, rotation=45, ha='right')
ax2.set_xlabel('Age Segment', fontsize=11, fontweight='bold')
ax2.set_ylabel('Transaction Count', fontsize=11, fontweight='bold')
ax2.set_title('Transactions by Age Segment', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('outputs/data_quality/4_customer_behavior.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 4_customer_behavior.png")
plt.close()

# ============================================================================
# VISUALIZATION 5: Media Spend Distribution
# ============================================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Spend by channel
channel_spend = media_df.groupby('channel')['spend_usd'].sum().sort_values(ascending=False)
colors = ['#006E74', '#0097AC', '#FF6B00', '#E8E8E8']
ax1.bar(range(len(channel_spend)), channel_spend.values, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(len(channel_spend)))
ax1.set_xticklabels(channel_spend.index, rotation=45, ha='right')
ax1.set_ylabel('Total Spend ($)', fontsize=11, fontweight='bold')
ax1.set_title('Total Media Spend by Channel', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

# Spend over time
test_media = media_df[media_df['week'].between(metadata['pre_period_weeks']+1, 
                                                 metadata['pre_period_weeks']+metadata['test_period_weeks'])]
treatment_media = test_media[test_media['geo_id'].isin(metadata['treatment_dmas'])].groupby('week')['spend_usd'].sum()
control_media = test_media[~test_media['geo_id'].isin(metadata['treatment_dmas'])].groupby('week')['spend_usd'].sum()

ax2.plot(treatment_media.index, treatment_media.values, label='Treatment', 
        marker='o', linewidth=2, color='#006E74')
ax2.plot(control_media.index, control_media.values, label='Control', 
        marker='s', linewidth=2, color='#FF6B00')
ax2.set_xlabel('Week', fontsize=11, fontweight='bold')
ax2.set_ylabel('Total Spend ($)', fontsize=11, fontweight='bold')
ax2.set_title('Media Spend During Test Period', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}K'))

plt.tight_layout()
plt.savefig('outputs/data_quality/5_media_spend.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 5_media_spend.png")
plt.close()

# ============================================================================
# VISUALIZATION 6: Sales Variability Analysis
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sales distribution
ax = axes[0, 0]
ax.hist(sales_df['sales_revenue'], bins=50, color='#0097AC', edgecolor='black', alpha=0.7)
ax.set_xlabel('Sales Revenue ($)', fontsize=10, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=10, fontweight='bold')
ax.set_title('Sales Revenue Distribution', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.axvline(sales_df['sales_revenue'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
ax.axvline(sales_df['sales_revenue'].median(), color='orange', linestyle='--', linewidth=2, label='Median')
ax.legend()

# Sales by DMA (box plot)
ax = axes[0, 1]
treatment_dmas_sample = sales_df[sales_df['is_treatment']==True]['geo_id'].unique()[:5]
control_dmas_sample = sales_df[sales_df['is_treatment']==False]['geo_id'].unique()[:5]
sample_dmas = list(treatment_dmas_sample) + list(control_dmas_sample)

box_data = [sales_df[sales_df['geo_id']==dma]['sales_revenue'].values for dma in sample_dmas]
bp = ax.boxplot(box_data, labels=[dma.replace('DMA_', '') for dma in sample_dmas], patch_artist=True)

for i, patch in enumerate(bp['boxes']):
    if i < 5:
        patch.set_facecolor('#006E74')
    else:
        patch.set_facecolor('#FF6B00')

ax.set_xlabel('DMA', fontsize=10, fontweight='bold')
ax.set_ylabel('Sales Revenue ($)', fontsize=10, fontweight='bold')
ax.set_title('Sales Variability Across DMAs (Sample)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Transaction amount distribution
ax = axes[1, 0]
ax.hist(transactions_df['revenue'], bins=50, color='#FF6B00', edgecolor='black', alpha=0.7)
ax.set_xlabel('Transaction Revenue ($)', fontsize=10, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=10, fontweight='bold')
ax.set_title('Transaction Amount Distribution', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Units per transaction
ax = axes[1, 1]
units_dist = transactions_df['units'].value_counts().sort_index()
ax.bar(units_dist.index, units_dist.values, color='#0097AC', edgecolor='black', linewidth=1.5)
ax.set_xlabel('Units per Transaction', fontsize=10, fontweight='bold')
ax.set_ylabel('Frequency', fontsize=10, fontweight='bold')
ax.set_title('Units per Transaction Distribution', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add percentages
total = units_dist.sum()
for i, (units, count) in enumerate(units_dist.items()):
    pct = count / total * 100
    ax.text(units, count + 20, f'{pct:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/data_quality/6_variability_analysis.png', dpi=300, bbox_inches='tight')
print("✅ Saved: 6_variability_analysis.png")
plt.close()

# ============================================================================
# SUMMARY REPORT
# ============================================================================

print("\n" + "=" * 80)
print("7️⃣ SUMMARY REPORT")
print("=" * 80)

print(f"\n✅ DATA QUALITY: PASSED")
print(f"   - Completeness: 100% (no missing values)")
print(f"   - Variety: High (multiple channels, promos, customers)")
print(f"   - Period structure: Clear (Pre/Test/Post)")
print(f"   - Lift detection: Evident in test period")

print(f"\n📊 KEY STATISTICS:")
print(f"   - Total weeks: {metadata['n_weeks']}")
print(f"   - Markets (DMAs): {metadata['n_dmas']}")
print(f"   - Unique customers: {metadata['n_unique_customers']}")
print(f"   - Test period ROI: {metadata.get('test_period_roi', 0):.1f}%")
print(f"   - Sustained lift: {metadata.get('sustained_lift_pct', 0):.1f}%")

print(f"\n📈 VISUALIZATIONS SAVED:")
print(f"   Location: outputs/data_quality/")
print(f"   1. Sales over time (treatment vs control)")
print(f"   2. Promo distribution by period")
print(f"   3. Sales by channel across periods")
print(f"   4. Customer behavior analysis")
print(f"   5. Media spend distribution")
print(f"   6. Sales variability analysis")

print("\n" + "=" * 80)
print("✅ DATA QUALITY TEST COMPLETE!")
print("=" * 80)


1️⃣ Loading data...
✅ Data loaded successfully!
   Sales: 7,800 records
   Transactions: 6,049 records
   Media: 10,400 records
   Controls: 2,600 records

2️⃣ DATA COMPLETENESS CHECK

Sales Data:
   ✅ No missing values!

Transactions Data:
   ✅ No missing values!

Media Data:
   ✅ No missing values!

Controls Data:
   ✅ No missing values!

✅ ALL DATA 100% COMPLETE!

3️⃣ DATA VARIETY ANALYSIS

📊 Sales Data Variety:
   Unique DMAs: 50
   Unique weeks: 52
   Retail channels: 3
   Promo types: 3
   Sales range: $583 - $26,697
   Sales std dev: $4,835
   Coefficient of variation: 42.53%

💳 Transaction Data Variety:
   Unique customers: 1301
   Age segments: 6
   New customers: 820 (13.6%)
   Transaction units range: 1 - 5
   Price range: $10.50 - $15.00

📺 Media Data Variety:
   Media channels: 4
   Spend range: $255 - $966
   Spend std dev: $174

🎛️ Controls Data Variety:
   Holiday weeks: 300 (11.5%)
   Promo weeks: 403 (15.5%)
   Competitor promo weeks: 442 (17.0%)

4️⃣ PERIOD BREAKDOW

C:\Users\249881\AppData\Local\Temp\ipykernel_2216\2751705807.py:365: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(box_data, labels=[dma.replace('DMA_', '') for dma in sample_dmas], patch_artist=True)


✅ Saved: 6_variability_analysis.png

7️⃣ SUMMARY REPORT

✅ DATA QUALITY: PASSED
   - Completeness: 100% (no missing values)
   - Variety: High (multiple channels, promos, customers)
   - Period structure: Clear (Pre/Test/Post)
   - Lift detection: Evident in test period

📊 KEY STATISTICS:
   - Total weeks: 52
   - Markets (DMAs): 50
   - Unique customers: 1301
   - Test period ROI: 828.9%
   - Sustained lift: 13.0%

📈 VISUALIZATIONS SAVED:
   Location: outputs/data_quality/
   1. Sales over time (treatment vs control)
   2. Promo distribution by period
   3. Sales by channel across periods
   4. Customer behavior analysis
   5. Media spend distribution
   6. Sales variability analysis

✅ DATA QUALITY TEST COMPLETE!


In [70]:
# ============================================================================
# ROI VALIDATION
# ============================================================================

print("\n" + "=" * 80)
print("8️⃣ ROI VALIDATION - ENSURING POSITIVE ROI")
print("=" * 80)

# Get test period dates from metadata
test_start_date = pd.to_datetime(metadata['test_start_date'])
test_end_date = pd.to_datetime(metadata['test_end_date'])
post_start_date = pd.to_datetime(metadata['post_start_date'])

treatment_dmas = metadata['treatment_dmas']
n_treatment = len(treatment_dmas)
n_control = metadata['n_dmas'] - n_treatment

print(f"\n📊 Setup:")
print(f"   Treatment DMAs: {n_treatment}")
print(f"   Control DMAs: {n_control}")
print(f"   Test period: {test_start_date.date()} to {test_end_date.date()}")

# ============================================================================
# TEST PERIOD ROI
# ============================================================================

print(f"\n" + "="*80)
print("🧮 TEST PERIOD ROI CALCULATION")
print("="*80)

# 1. Calculate treatment sales
test_sales = sales_df[sales_df['is_test_period'] == True]
treatment_sales_test = test_sales[test_sales['is_treatment'] == True]['sales_revenue'].sum()
control_sales_test = test_sales[test_sales['is_treatment'] == False]['sales_revenue'].sum()

print(f"\n1️⃣ Sales Comparison:")
print(f"   Treatment total sales: ${treatment_sales_test:,.0f}")
print(f"   Control total sales: ${control_sales_test:,.0f}")

# 2. Scale control to treatment group size
scale_factor = n_treatment / n_control
control_sales_scaled = control_sales_test * scale_factor
print(f"\n2️⃣ Scaled Control (to match treatment size):")
print(f"   Scale factor: {scale_factor:.2f}")
print(f"   Scaled control sales: ${control_sales_scaled:,.0f}")

# 3. Calculate incremental sales
incremental_sales = treatment_sales_test - control_sales_scaled
print(f"\n3️⃣ Incremental Sales:")
print(f"   Incremental sales: ${incremental_sales:,.0f}")
print(f"   Lift %: {(incremental_sales / control_sales_scaled * 100):.1f}%")

# 4. Calculate incremental profit (50% margin)
profit_margin = 0.50
incremental_profit = incremental_sales * profit_margin
print(f"\n4️⃣ Incremental Profit:")
print(f"   Profit margin: {profit_margin*100:.0f}%")
print(f"   Incremental profit: ${incremental_profit:,.0f}")

# 5. Calculate media spend
test_media = media_df[
    (media_df['date'] >= test_start_date) &
    (media_df['date'] <= test_end_date) &
    (media_df['geo_id'].isin(treatment_dmas))
]
total_spend_test = test_media['spend_usd'].sum()

# Calculate extra spend (20% increase during test)
spend_increase_pct = 0.20
normal_spend = total_spend_test / (1 + spend_increase_pct)
extra_spend = total_spend_test - normal_spend

print(f"\n5️⃣ Media Spend:")
print(f"   Total treatment spend: ${total_spend_test:,.0f}")
print(f"   Normal spend (baseline): ${normal_spend:,.0f}")
print(f"   Extra spend (incremental): ${extra_spend:,.0f}")

# 6. Calculate ROI
roi = ((incremental_profit - extra_spend) / extra_spend) * 100 if extra_spend > 0 else 0
roas = incremental_profit / extra_spend if extra_spend > 0 else 0

print(f"\n6️⃣ ROI METRICS:")
print(f"   Profit ROAS: {roas:.2f}x")
print(f"   ROI: {roi:.1f}%")
print(f"   Net profit: ${(incremental_profit - extra_spend):,.0f}")

# Validate ROI is positive
if roi > 0:
    print(f"\n   ✅ TEST PERIOD ROI IS POSITIVE: {roi:.1f}%")
else:
    print(f"\n   ❌ WARNING: TEST PERIOD ROI IS NEGATIVE: {roi:.1f}%")

# ============================================================================
# POST PERIOD ROI
# ============================================================================

print(f"\n" + "="*80)
print("🧮 POST PERIOD ROI CALCULATION (Sustained Effect)")
print("="*80)

# 1. Calculate treatment sales
post_sales = sales_df[sales_df['is_post_period'] == True]
treatment_sales_post = post_sales[post_sales['is_treatment'] == True]['sales_revenue'].sum()
control_sales_post = post_sales[post_sales['is_treatment'] == False]['sales_revenue'].sum()

print(f"\n1️⃣ Sales Comparison:")
print(f"   Treatment total sales: ${treatment_sales_post:,.0f}")
print(f"   Control total sales: ${control_sales_post:,.0f}")

# 2. Scale control
control_sales_scaled_post = control_sales_post * scale_factor
print(f"\n2️⃣ Scaled Control:")
print(f"   Scaled control sales: ${control_sales_scaled_post:,.0f}")

# 3. Calculate incremental sales
incremental_sales_post = treatment_sales_post - control_sales_scaled_post
sustained_lift_pct = (incremental_sales_post / control_sales_scaled_post * 100) if control_sales_scaled_post > 0 else 0

print(f"\n3️⃣ Incremental Sales:")
print(f"   Incremental sales: ${incremental_sales_post:,.0f}")
print(f"   Sustained lift %: {sustained_lift_pct:.1f}%")

# 4. Calculate incremental profit
incremental_profit_post = incremental_sales_post * profit_margin
print(f"\n4️⃣ Incremental Profit:")
print(f"   Incremental profit: ${incremental_profit_post:,.0f}")

# 5. Media spend (back to normal, no extra spend)
print(f"\n5️⃣ Media Spend:")
print(f"   Extra spend: $0 (marketing back to normal)")
print(f"   Pure profit from sustained behavior change!")

if incremental_profit_post > 0:
    print(f"\n   ✅ POST PERIOD PROFIT IS POSITIVE: ${incremental_profit_post:,.0f}")
else:
    print(f"\n   ❌ WARNING: POST PERIOD PROFIT IS NEGATIVE: ${incremental_profit_post:,.0f}")

# ============================================================================
# COMBINED ROI
# ============================================================================

print(f"\n" + "="*80)
print("📊 COMBINED ROI SUMMARY")
print("="*80)

total_incremental_profit = incremental_profit + incremental_profit_post
total_extra_spend = extra_spend  # Only test period had extra spend

combined_roi = ((total_incremental_profit - total_extra_spend) / total_extra_spend) * 100 if total_extra_spend > 0 else 0
combined_roas = total_incremental_profit / total_extra_spend if total_extra_spend > 0 else 0

print(f"\nTest + Post Period Combined:")
print(f"   Total incremental profit: ${total_incremental_profit:,.0f}")
print(f"   Total extra spend: ${total_extra_spend:,.0f}")
print(f"   Combined Profit ROAS: {combined_roas:.2f}x")
print(f"   Combined ROI: {combined_roi:.1f}%")
print(f"   Net profit: ${(total_incremental_profit - total_extra_spend):,.0f}")

# ============================================================================
# COMPARISON WITH METADATA
# ============================================================================

print(f"\n" + "="*80)
print("🔍 VALIDATION AGAINST METADATA")
print("="*80)

metadata_roi = metadata.get('test_period_roi', 0)
metadata_roas = metadata.get('test_period_roas', 0)

print(f"\nTest Period Comparison:")
print(f"   Calculated ROI: {roi:.1f}%")
print(f"   Metadata ROI: {metadata_roi:.1f}%")
print(f"   Match: {'✅ YES' if abs(roi - metadata_roi) < 1 else '❌ NO'}")
print(f"\n   Calculated ROAS: {roas:.2f}x")
print(f"   Metadata ROAS: {metadata_roas:.2f}x")
print(f"   Match: {'✅ YES' if abs(roas - metadata_roas) < 0.1 else '❌ NO'}")

# ============================================================================
# FINAL VALIDATION
# ============================================================================

print(f"\n" + "="*80)
print("✅ FINAL ROI VALIDATION")
print("="*80)

all_positive = True

print(f"\n✓ Checks:")
if roi > 0:
    print(f"   ✅ Test period ROI is positive: {roi:.1f}%")
else:
    print(f"   ❌ Test period ROI is NEGATIVE: {roi:.1f}%")
    all_positive = False

if roas > 1:
    print(f"   ✅ Test period ROAS > 1: {roas:.2f}x")
else:
    print(f"   ❌ Test period ROAS < 1: {roas:.2f}x")
    all_positive = False

if incremental_profit_post > 0:
    print(f"   ✅ Post period has sustained profit: ${incremental_profit_post:,.0f}")
else:
    print(f"   ⚠️  Post period profit is negative (check sustained effect)")

if combined_roi > 0:
    print(f"   ✅ Combined ROI is positive: {combined_roi:.1f}%")
else:
    print(f"   ❌ Combined ROI is NEGATIVE: {combined_roi:.1f}%")
    all_positive = False

if all_positive:
    print(f"\n{'='*80}")
    print("🎉 SUCCESS: ALL ROI METRICS ARE POSITIVE!")
    print("="*80)
    print(f"\nKey Takeaways:")
    print(f"   • Every $1 of extra marketing spend generated ${roas:.2f} in profit")
    print(f"   • ROI of {roi:.1f}% means {roi/100:.1f}x return on investment")
    print(f"   • Sustained {sustained_lift_pct:.1f}% lift after campaign ended")
    print(f"   • Total net profit: ${(total_incremental_profit - total_extra_spend):,.0f}")
else:
    print(f"\n{'='*80}")
    print("⚠️  WARNING: SOME ROI METRICS ARE NOT POSITIVE")
    print("="*80)
    print(f"\nThis indicates potential data issues. Check:")
    print(f"   • Treatment effect sizes (price_effect, visibility_effect)")
    print(f"   • Media spend levels during test period")
    print(f"   • Profit margin assumptions")


8️⃣ ROI VALIDATION - ENSURING POSITIVE ROI

📊 Setup:
   Treatment DMAs: 35
   Control DMAs: 15
   Test period: 2024-07-21 to 2024-10-06

🧮 TEST PERIOD ROI CALCULATION

1️⃣ Sales Comparison:
   Treatment total sales: $15,955,754
   Control total sales: $5,502,583

2️⃣ Scaled Control (to match treatment size):
   Scale factor: 2.33
   Scaled control sales: $12,839,361

3️⃣ Incremental Sales:
   Incremental sales: $3,116,393
   Lift %: 24.3%

4️⃣ Incremental Profit:
   Profit margin: 50%
   Incremental profit: $1,558,196

5️⃣ Media Spend:
   Total treatment spend: $1,006,499
   Normal spend (baseline): $838,749
   Extra spend (incremental): $167,750

6️⃣ ROI METRICS:
   Profit ROAS: 9.29x
   ROI: 828.9%
   Net profit: $1,390,446

   ✅ TEST PERIOD ROI IS POSITIVE: 828.9%

🧮 POST PERIOD ROI CALCULATION (Sustained Effect)

1️⃣ Sales Comparison:
   Treatment total sales: $17,134,267
   Control total sales: $6,499,808

2️⃣ Scaled Control:
   Scaled control sales: $15,166,218

3️⃣ Incremental 